# Full Model Evaluation — All Configurations

Evaluates every trained model variant and saves results to `results/all_results.json`.

**Coverage:** 9 no-index baselines (3 models × 3 datasets) + 27 RAC models
(3 models × 3 index types × 3 datasets) = 36 total configurations.

## 1. Imports

In [ ]:
import sys
import json
from pathlib import Path
import torch
import faiss
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path("..").resolve()))
from retriever import retrieve_top_k_above_threshold
from data_loaders import load_ihc_binary, load_ishate_binary, load_vicomtech
from training_utils import tokenize_augmented, filter_records

## 2. Configuration

`BEST_PARAMS` holds the optimal retrieval hyperparameters found during HP tuning.
Augmentation is cached at the maximum K and minimum threshold, then filtered
per model to avoid redundant encoding.

In [ ]:
# Define paths
ROOT_DIR        = Path("../..")
INDEX_DIR       = ROOT_DIR / "corpus" / "index"
WEIGHTS_DIR     = ROOT_DIR / "weigths" / "weights_baseline"
WEIGHTS_RAC_DIR = ROOT_DIR / "weigths" / "weights_rac_best_hyperparameters"
RESULTS_FILE    = ROOT_DIR / "new_results" / "all_results.json"

RETRIEVER_HF_ID = "sentence-transformers/all-mpnet-base-v2"

# Best HP per model from tuning
BEST_PARAMS = {
    'bert':     {'k': 5, 'threshold': 0.5},
    'hatebert': {'k': 3, 'threshold': 0.4},
    'roberta':  {'k': 3, 'threshold': 0.4},
}

# Cache bounds
MAX_K         = max(p['k']         for p in BEST_PARAMS.values())
MIN_THRESHOLD = min(p['threshold'] for p in BEST_PARAMS.values())
MAX_LENGTH    = 256
BATCH_SIZE    = 64

# Models, index types and datasets
MODEL_MAP  = {'bert': 'bert-base-uncased', 'hatebert': 'GroNLP/hateBERT', 'roberta': 'roberta-base'}
INDEX_TYPES = ['example', 'knowledge', 'full']
DATASETS    = ['IHC', 'ISHate', 'Vicomtech']
MODELS      = ['bert', 'hatebert', 'roberta']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 3. Load Test Datasets

Load IHC, ISHate and Vicomtech test splits using `data_loaders.py`.

In [ ]:
# Load the 3 test datasets
_, test_ihc    = load_ihc_binary(seed=42)
_, test_ishate = load_ishate_binary()
test_vicomtech = load_vicomtech(split='test')

DATASET_MAP = {
    'IHC':       {'test': test_ihc,       'text_col': 'post'},
    'ISHate':    {'test': test_ishate,     'text_col': 'text'},
    'Vicomtech': {'test': test_vicomtech,  'text_col': 'text'},
}
print(f'IHC: {len(test_ihc):,}  ISHate: {len(test_ishate):,}  Vicomtech: {len(test_vicomtech):,}')

## 4. Evaluation Helpers

`compute_metrics_from_arrays` computes macro F1/P/R from raw prediction arrays.
`augment_test` retrieves neighbors for a test split (no self-exclusion needed).
`evaluate_augmented` runs batched inference on pre-augmented records.
`tokenize_augmented` and `filter_records` are imported from `training_utils`.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# compute_metrics_from_arrays: compute F1/P/R from raw arrays
def compute_metrics_from_arrays(preds, labels):
    return {
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro', zero_division=0),
    }

# augment_test: retrieve neighbors for a test split; store (text, score) pairs so
# filter_records can apply per-model k and threshold without re-encoding
def augment_test(hf_dataset, text_col, ret_model, ret_tokenizer, ret_index, ret_documents, k, threshold):
    records = []
    for example in tqdm(hf_dataset, desc='augment', leave=False):
        tweet     = example[text_col]
        neighbors = retrieve_top_k_above_threshold(
            tweet, threshold, ret_model, ret_tokenizer, ret_index, ret_documents,
            chunk_id=None, k=k, use_mean_pool=True,
        )
        records.append({'query': tweet, 'neighbors': neighbors, 'label': example['label']})
    return records

# evaluate_augmented: run batched inference on pre-augmented records
def evaluate_augmented(clf_model, clf_tokenizer, records):
    tok = tokenize_augmented(records, clf_tokenizer, max_length=MAX_LENGTH)
    clf_model.eval()
    all_preds = []
    for i in range(0, len(tok), BATCH_SIZE):
        batch          = tok[i:i+BATCH_SIZE]
        input_ids      = torch.tensor(batch['input_ids']).to(device)
        attention_mask = torch.tensor(batch['attention_mask']).to(device)
        with torch.no_grad():
            logits = clf_model(input_ids=input_ids, attention_mask=attention_mask).logits
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
    return compute_metrics_from_arrays(all_preds, tok['labels'])

## 5. No-Index Baselines

Run plain inference (no retrieval) on all 9 combinations of model × dataset.
Weights loaded from `weigths/weights_baseline/{model}/{dataset}/`.

In [ ]:
results = {}

# Load model and run plain inference for each (model, dataset)
for model_key in MODELS:
    for ds_name in DATASETS:
        weight_path = WEIGHTS_DIR / model_key / ds_name
        print(f'  {model_key} | no-index | {ds_name} ...', end=' ', flush=True)

        tokenizer = AutoTokenizer.from_pretrained(str(weight_path))
        model     = AutoModelForSequenceClassification.from_pretrained(str(weight_path)).to(device)
        ds_cfg    = DATASET_MAP[ds_name]

        texts  = [ex[ds_cfg['text_col']] for ex in ds_cfg['test']]
        labels = [ex['label'] for ex in ds_cfg['test']]
        all_preds = []
        model.eval()
        for i in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[i:i+BATCH_SIZE]
            inputs = tokenizer(batch_texts, truncation=True, padding=True,
                               max_length=MAX_LENGTH, return_tensors='pt').to(device)
            with torch.no_grad():
                logits = model(**inputs).logits
            all_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())

        metrics = compute_metrics_from_arrays(all_preds, labels)
        results[(model_key, 'no-index', ds_name)] = metrics
        print(f'F1={metrics["macro_f1"]:.4f}')

        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

## 6. RAC Models

Load the SBERT retriever once, then loop over the 3 index types.
For each index, augment all datasets (cached at MAX_K / MIN_THRESHOLD),
then evaluate each (model, dataset) pair using the model-specific (k, threshold).

In [ ]:
# Load the sbert retriever
print(f'Loading retriever: {RETRIEVER_HF_ID} ...')
ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)
print('Retriever ready.\n')

# Loop over index types
for index_type in INDEX_TYPES:
    ret_index = faiss.read_index(str(INDEX_DIR / f'vdb_{index_type}.faiss'))
    with open(INDEX_DIR / f'lookup_{index_type}.json') as f:
        ret_documents = json.load(f)
    print(f'\n--- Index: {index_type} ({ret_index.ntotal:,} vectors) ---')

    # Augment all datasets once per index
    aug_cache = {}
    for ds_name in DATASETS:
        ds_cfg = DATASET_MAP[ds_name]
        aug_cache[ds_name] = augment_test(
            ds_cfg['test'], ds_cfg['text_col'],
            ret_model, ret_tokenizer, ret_index, ret_documents,
            k=MAX_K, threshold=MIN_THRESHOLD,
        )

    # Evaluate each (model, dataset) pair
    for model_key in MODELS:
        k         = BEST_PARAMS[model_key]['k']
        threshold = BEST_PARAMS[model_key]['threshold']

        for ds_name in DATASETS:
            weight_path = WEIGHTS_RAC_DIR / model_key / 'sbert' / index_type / ds_name
            print(f'  {model_key} | {index_type} | {ds_name} ...', end=' ', flush=True)

            clf_tokenizer = AutoTokenizer.from_pretrained(str(weight_path))
            clf_model     = AutoModelForSequenceClassification.from_pretrained(
                str(weight_path), num_labels=2).to(device)

            filtered = filter_records(aug_cache[ds_name], k=k, threshold=threshold)
            metrics  = evaluate_augmented(clf_model, clf_tokenizer, filtered)
            results[(model_key, index_type, ds_name)] = metrics
            print(f'F1={metrics["macro_f1"]:.4f}')

            del clf_model
            if device.type == 'cuda':
                torch.cuda.empty_cache()

del ret_model
if device.type == 'cuda':
    torch.cuda.empty_cache()

## 7. Save and Display Results

Results are saved to `results/all_results.json`.
The summary table shows macro F1 for every configuration.

In [ ]:
# Save to JSON
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
serializable = {f'{m}|{it}|{ds}': v for (m, it, ds), v in results.items()}
with open(RESULTS_FILE, 'w') as f:
    json.dump(serializable, f, indent=2)
print(f'Saved to {RESULTS_FILE}')

# Print text summary
print('\nSummary (Macro F1):')
header = f'{"Model":<10} {"Index":<12}  {"IHC":>6}  {"ISHate":>7}  {"Vicomtech":>10}'
print(header)
print('-' * len(header))
for model_key in MODELS:
    for index_type in ['no-index'] + INDEX_TYPES:
        row = f'{model_key:<10} {index_type:<12}'
        for ds_name in DATASETS:
            v = results.get((model_key, index_type, ds_name), {}).get('macro_f1', float('nan'))
            row += f'  {v:>6.4f}'
        print(row)
    print()

In [ ]:
# Build and display styled DataFrame
import pandas as pd

rows = []
for model_key in MODELS:
    for index_type in ['no-index'] + INDEX_TYPES:
        row = {'Model': model_key, 'Index': index_type}
        for ds_name in DATASETS:
            row[ds_name] = results.get((model_key, index_type, ds_name), {}).get('macro_f1', float('nan'))
        rows.append(row)

df = pd.DataFrame(rows).set_index(['Model', 'Index'])
df.style.format('{:.4f}').highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')